# Projeto Final – Processamento de Imagens
## Classificação de imagens com Abordagem Clássica: **GLCM (Haralick) + Filtros de Gabor** + SVM / Random Forest

**Aluna:** Lívia Pontes Argenton  
**Disciplina:** Processamento de Imagens – UTFPR-CP (Prof. Claiton de Oliveira)  
**Dataset:** Covid-19 & Normal (raio-X de tórax, escala de cinza)

### Pipeline
Aquisição → Pré-processamento → **Extração de características (GLCM + Gabor)** → Classificação (SVM / Random Forest) → Avaliação

### Por que estes extratores são DIFERENTES dos vistos em aula?
Em aula foram apresentados: **SIFT + Bag of Visual Words**, **HOG**, **LBP** e **histogramas de cor (GCH/LCH)**.  
Aqui são usados:
- **GLCM / Haralick** (estatísticas de segunda ordem da textura, a partir da matriz de co-ocorrência de níveis de cinza);
- **Banco de filtros de Gabor** (textura no domínio espaço-frequência, com várias orientações e escalas).

## 1. Bibliotecas

In [ ]:
import os, glob, time, warnings
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from joblib import Parallel, delayed

from skimage.feature import graycomatrix, graycoprops

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, classification_report, accuracy_score,
                             precision_recall_fscore_support, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
print("OpenCV:", cv2.__version__)

## 2. Configuração e carregamento do dataset

O dataset deve estar em uma pasta com **uma subpasta por classe**, por exemplo:

```
dataset/
 ├── COVID/    (imagens .png/.jpg)
 └── Normal/
```

No Colab, o caminho mais simples é copiar o dataset para o Google Drive e montá-lo. Ajuste `DATASET_DIR` abaixo.

In [ ]:
# ---- CONFIGURAÇÕES (ajuste aqui) ----
USAR_DRIVE   = True                         # True = monta o Google Drive no Colab
DATASET_DIR  = "/content/drive/MyDrive/dataset"   # pasta com uma subpasta por classe
IMG_SIZE     = 224                          # as imagens são redimensionadas para IMG_SIZE x IMG_SIZE
MAX_POR_CLASSE = None                       # None = usa todas; ou ex.: 1000 para limitar
TEST_SIZE    = 0.20                         # holdout 80/20 (como nos slides)
N_JOBS       = -1                           # usa todos os núcleos disponíveis

if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        print("Fora do Colab: ignorando a montagem do Drive.")

In [ ]:
EXTS = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff")

def listar_imagens(dataset_dir, max_por_classe=None):
    classes = sorted(d for d in os.listdir(dataset_dir)
                     if os.path.isdir(os.path.join(dataset_dir, d)))
    caminhos, rotulos = [], []
    rng = np.random.RandomState(SEED)
    for c in classes:
        arqs = []
        for e in EXTS:
            arqs += glob.glob(os.path.join(dataset_dir, c, "**", e), recursive=True)
        arqs = sorted(arqs)
        if max_por_classe and len(arqs) > max_por_classe:
            arqs = list(rng.choice(arqs, max_por_classe, replace=False))
        caminhos += arqs
        rotulos += [c] * len(arqs)
    return np.array(caminhos), np.array(rotulos), classes

caminhos, rotulos_txt, classes = listar_imagens(DATASET_DIR, MAX_POR_CLASSE)
le = LabelEncoder()
y = le.fit_transform(rotulos_txt)
print("Classes:", list(le.classes_))
print("Total de imagens:", len(caminhos))
print(pd.Series(rotulos_txt).value_counts().to_string())

## 3. Pré-processamento
Cada imagem é lida em **escala de cinza**, redimensionada para 224×224 e suavizada com filtro de mediana 3×3 (reduz ruído, como feito nos códigos de aula).  
A equalização local (CLAHE) é opcional e ajuda a realçar a textura nos raios-X.

In [ ]:
USAR_CLAHE = True
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocessar(caminho, tamanho=IMG_SIZE):
    img = cv2.imread(caminho, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Não foi possível ler: {caminho}")
    img = cv2.resize(img, (tamanho, tamanho), interpolation=cv2.INTER_AREA)
    img = cv2.medianBlur(img, 3)
    if USAR_CLAHE:
        img = clahe.apply(img)
    return img

# Visualização de exemplos de cada classe
fig, axs = plt.subplots(len(le.classes_), 4, figsize=(12, 3 * len(le.classes_)))
axs = np.atleast_2d(axs)
for i, c in enumerate(le.classes_):
    idx = np.where(rotulos_txt == c)[0][:4]
    for j, k in enumerate(idx):
        axs[i, j].imshow(preprocessar(caminhos[k]), cmap="gray")
        axs[i, j].set_title(c); axs[i, j].axis("off")
plt.tight_layout(); plt.show()

## 4. Divisão treino/teste (Holdout 80/20 estratificado)
`stratify=y` garante todas as classes em ambos os conjuntos, com a mesma proporção.

In [ ]:
X_tr_paths, X_te_paths, y_train, y_test = train_test_split(
    caminhos, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)

print("Treino:", len(X_tr_paths), "| Teste:", len(X_te_paths))
print("Treino por classe:", dict(zip(le.classes_, np.bincount(y_train))))
print("Teste  por classe:", dict(zip(le.classes_, np.bincount(y_test))))

## 5. Extração de características

### 5.1 GLCM / Haralick (textura – estatísticas de 2ª ordem)
1. A imagem é quantizada para **32 níveis de cinza** (reduz o custo e a esparsidade da matriz).
2. Calcula-se a GLCM para as distâncias **1, 3 e 5** e os ângulos **0°, 45°, 90° e 135°**.
3. De cada GLCM extraem-se: *contraste, dissimilaridade, homogeneidade, energia, correlação, ASM* e **entropia**.
4. Para cada distância, tira-se a **média entre os 4 ângulos** (torna o descritor aproximadamente invariante à rotação).

Tamanho do vetor: 3 distâncias × 7 propriedades = **21 características**.

### 5.2 Banco de filtros de Gabor (textura – espaço/frequência)
- **4 orientações** (0°, 45°, 90°, 135°) × **3 comprimentos de onda** (4, 8, 16 px) = **12 filtros**.
- Para cada resposta: **média, desvio padrão e energia** (média dos quadrados) da magnitude.

Tamanho do vetor: 12 × 3 = **36 características**.

**Vetor final (GLCM + Gabor) = 57 características por imagem** (bem menor que os 7056 do HOG visto em aula).

In [ ]:
# ---------- GLCM / Haralick ----------
NIVEIS = 32
DISTANCIAS = [1, 3, 5]
ANGULOS = [0, np.pi/4, np.pi/2, 3*np.pi/4]
PROPS = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]

def extrair_glcm_img(img):
    q = (img.astype(np.float32) / 256.0 * NIVEIS).astype(np.uint8)       # quantiza para 0..31
    glcm = graycomatrix(q, distances=DISTANCIAS, angles=ANGULOS,
                        levels=NIVEIS, symmetric=True, normed=True)
    feats = []
    for d in range(len(DISTANCIAS)):
        for p in PROPS:
            feats.append(graycoprops(glcm, p)[d].mean())                  # média sobre os ângulos
        # entropia: -sum(p * log2 p), média sobre os ângulos
        ent = []
        for a in range(len(ANGULOS)):
            P = glcm[:, :, d, a]
            P = P[P > 0]
            ent.append(-np.sum(P * np.log2(P)))
        feats.append(np.mean(ent))
    return np.array(feats, dtype=np.float32)

# ---------- Banco de filtros de Gabor ----------
ORIENTACOES = [0, np.pi/4, np.pi/2, 3*np.pi/4]
LAMBDAS = [4, 8, 16]

def criar_banco_gabor():
    banco = []
    for lam in LAMBDAS:
        for th in ORIENTACOES:
            k = cv2.getGaborKernel(ksize=(31, 31), sigma=0.56 * lam, theta=th,
                                   lambd=lam, gamma=0.5, psi=0, ktype=cv2.CV_32F)
            k /= (np.abs(k).sum() + 1e-8)                                 # normaliza energia do kernel
            banco.append(k)
    return banco

BANCO_GABOR = criar_banco_gabor()

def extrair_gabor_img(img):
    f = img.astype(np.float32) / 255.0
    feats = []
    for k in BANCO_GABOR:
        resp = np.abs(cv2.filter2D(f, cv2.CV_32F, k))
        feats += [resp.mean(), resp.std(), (resp ** 2).mean()]
    return np.array(feats, dtype=np.float32)

def extrair_tudo(caminho):
    img = preprocessar(caminho)
    return extrair_glcm_img(img), extrair_gabor_img(img)

def extrair_features(lista_caminhos):
    res = Parallel(n_jobs=N_JOBS, verbose=0)(delayed(extrair_tudo)(c) for c in lista_caminhos)
    glcm = np.vstack([r[0] for r in res])
    gabor = np.vstack([r[1] for r in res])
    return glcm, gabor

In [ ]:
t0 = time.time()
G_tr, B_tr = extrair_features(X_tr_paths)
G_te, B_te = extrair_features(X_te_paths)
print(f"Extração concluída em {time.time() - t0:.1f}s")
print("GLCM  :", G_tr.shape, "| Gabor:", B_tr.shape)

# Conjuntos de características (para o estudo de ablação)
FEATURES = {
    "GLCM":          (G_tr, G_te),
    "Gabor":         (B_tr, B_te),
    "GLCM + Gabor":  (np.hstack([G_tr, B_tr]), np.hstack([G_te, B_te])),
}

## 6. Treinamento e classificação (SVM e Random Forest)

In [ ]:
# O StandardScaler fica DENTRO do Pipeline: assim ele é ajustado só com os dados de
# treino de cada dobra do cross-validation (sem vazamento de informação do teste).
MODELOS = {
    "SVM": (
        Pipeline([("sc", StandardScaler()), ("clf", SVC(class_weight="balanced", random_state=SEED))]),
        {"clf__kernel": ["rbf"], "clf__C": [0.1, 1, 10, 100], "clf__gamma": ["scale", 0.01, 0.001]},
    ),
    "Random Forest": (
        Pipeline([("sc", StandardScaler()), ("clf", RandomForestClassifier(class_weight="balanced",
                                                                         random_state=SEED, n_jobs=-1))]),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 10, 20],
         "clf__min_samples_leaf": [1, 3]},
    ),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def treinar_avaliar(nome_feat, nome_modelo):
    Xtr, Xte = FEATURES[nome_feat]
    pipe, grade = MODELOS[nome_modelo]
    gs = GridSearchCV(pipe, grade, cv=cv, scoring="f1_macro", n_jobs=-1)
    gs.fit(Xtr, y_train)
    y_pred = gs.predict(Xte)
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
    return {"features": nome_feat, "modelo": nome_modelo,
            "melhores_params": gs.best_params_, "cv_f1_macro": gs.best_score_,
            "acuracia": accuracy_score(y_test, y_pred),
            "precisao_macro": p, "recall_macro": r, "f1_macro": f1,
            "y_pred": y_pred}

resultados = []
for nf in FEATURES:
    for nm in MODELOS:
        t0 = time.time()
        res = treinar_avaliar(nf, nm)
        resultados.append(res)
        print(f"{nf:14s} | {nm:14s} | acc={res['acuracia']:.4f} | F1(macro)={res['f1_macro']:.4f} | {time.time()-t0:.1f}s")

## 7. Comparação dos resultados (estudo de ablação)
Compara cada extrator isolado e combinado, com os dois classificadores.

In [ ]:
tabela = pd.DataFrame([{k: v for k, v in r.items() if k not in ("y_pred", "melhores_params")} for r in resultados])
tabela = tabela.sort_values("f1_macro", ascending=False).reset_index(drop=True)
display(tabela.round(4))

melhor = max(resultados, key=lambda r: r["f1_macro"])
print(f"\nMELHOR CONFIGURAÇÃO: {melhor['features']} + {melhor['modelo']}")
print("Hiperparâmetros:", melhor["melhores_params"])

## 8. Resultado final – melhor configuração
Matriz de confusão, precisão, recall, F1-score, suporte e acurácia no conjunto de **teste** (20% dos dados, nunca usado no treino nem na escolha de hiperparâmetros).

In [ ]:
y_pred = melhor["y_pred"]

# Matriz de confusão (linhas = rótulos reais, colunas = rótulos previstos)
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel("Rótulos previstos"); ax.set_ylabel("Rótulos reais")
ax.set_title(f"Matriz de confusão – {melhor['features']} + {melhor['modelo']}")
plt.tight_layout(); plt.show()

# Precisão, recall, F1-score e suporte por classe
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=4))
print(f"ACURÁCIA: {accuracy_score(y_test, y_pred):.4f}")

## 9. Matrizes de confusão de todas as configurações

In [ ]:
n = len(resultados)
cols = 3
linhas = int(np.ceil(n / cols))
fig, axs = plt.subplots(linhas, cols, figsize=(5 * cols, 4.2 * linhas))
axs = np.array(axs).ravel()
for ax, r in zip(axs, resultados):
    ConfusionMatrixDisplay.from_predictions(y_test, r["y_pred"], display_labels=le.classes_,
                                            cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(f"{r['features']} + {r['modelo']}\nacc={r['acuracia']:.3f}", fontsize=10)
for ax in axs[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 10. Visualização dos descritores (apoio ao vídeo)
Filtros de Gabor usados e respostas em uma imagem de teste.

In [ ]:
# Banco de filtros
fig, axs = plt.subplots(len(LAMBDAS), len(ORIENTACOES), figsize=(8, 6))
for i, lam in enumerate(LAMBDAS):
    for j, th in enumerate(ORIENTACOES):
        axs[i, j].imshow(BANCO_GABOR[i * len(ORIENTACOES) + j], cmap="gray")
        axs[i, j].set_title(f"λ={lam}, θ={int(np.degrees(th))}°", fontsize=8); axs[i, j].axis("off")
plt.suptitle("Banco de filtros de Gabor"); plt.tight_layout(); plt.show()

# Respostas em uma imagem de exemplo
img = preprocessar(X_te_paths[0])
f = img.astype(np.float32) / 255.0
fig, axs = plt.subplots(1, 4, figsize=(14, 3.6))
axs[0].imshow(img, cmap="gray"); axs[0].set_title("Imagem pré-processada")
for ax, idx in zip(axs[1:], [0, 5, 10]):
    ax.imshow(np.abs(cv2.filter2D(f, cv2.CV_32F, BANCO_GABOR[idx])), cmap="magma")
    ax.set_title(f"Resposta Gabor #{idx}")
for ax in axs: ax.axis("off")
plt.tight_layout(); plt.show()

## 11. Resumo para o README

In [ ]:
print("Dataset         :", DATASET_DIR)
print("Classes         :", list(le.classes_))
print("Treino / Teste  :", len(X_tr_paths), "/", len(X_te_paths), "(holdout 80/20 estratificado)")
print("Extratores      : GLCM/Haralick (21) + Gabor (36) = 57 características")
print("Melhor modelo   :", melhor["features"], "+", melhor["modelo"])
print(f"Acurácia (teste): {melhor['acuracia']:.4f}")
print(f"F1 macro        : {melhor['f1_macro']:.4f}")